# 10s Best Selector: VBS Gap and Wilcoxon Test

This notebook analyzes the best 10s selector against SBS and VBS.

Definitions used here:

- Lower cost and lower average tied rank are better. Ranks are recomputed from `label_costs` using standard average-rank tie handling.
- Best selector decision: solver with minimum `pred_costs`.
- VBS cost: minimum true `label_costs` per instance.
- SBS: one fixed solver with lowest mean true cost over the test set.
- Selector VBS gap: `selector_cost - vbs_cost`.
- SBS VBS gap: `sbs_cost - vbs_cost`.
- Per-instance VBS gap closed ratio: `(sbs_cost - selector_cost) / (sbs_cost - vbs_cost)`.
- Aggregate VBS gap closed ratio: `(mean_sbs_cost - mean_selector_cost) / (mean_sbs_cost - mean_vbs_cost)`.

A ratio of 0 means no improvement over SBS, 1 means the VBS gap is fully closed, and values above 1 mean the selector beats VBS on the normalized metric only if numerical/tie effects appear; in normal lower-bound terms VBS is best, so above 1 should be inspected.

Note: the aggregate gap-closed ratio is usually the most stable number. The mean of per-instance gap-closed ratios can be noisy when `sbs_cost - vbs_cost` is very close to zero, because the denominator is tiny. The notebook reports both the raw per-instance mean and an epsilon-filtered version.


In [ ]:
from pathlib import Path
import json
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import wilcoxon

RESULTS_ROOT = Path('10s_v2')
PREDICTION_FILE = 'predictions_epoch30.json'
SOLVERS = ['CLK', 'EAX', 'LKH', 'MAOS', 'CONCORDE']

# Set this to None to auto-pick one overall selector by cost, with rank as the tie-breaker.
# You can also set it manually, e.g. 'MAE+LambdaRank lw=0.5'.
BEST_SELECTOR = None
GAP_EPS = 1e-8

pd.set_option('display.max_rows', 80)
pd.set_option('display.max_columns', 80)
pd.set_option('display.width', 180)


In [ ]:
def parse_selector_metadata(folder_name: str) -> dict:
    base_match = re.search(
        r'results_(?P<cutoff>[0-9.]+s)_(?P<hidden>\d+)hd_(?P<lr>[^_]+)lr_(?P<dr>[^_]+)dr_'
        r'(?P<cost_loss>MSE|MAE|Huber)_(?P<rank_loss>RankNet|ListNet|LambdaRank)_'
        r'(?P<rank_method>[^_]+)_(?P<loss_weight>[0-9.]+)lw(?P<suffix>.*)$',
        folder_name,
    )
    if not base_match:
        return {'method': folder_name, 'objective_type': 'unknown'}

    meta = base_match.groupdict()
    suffix = meta['suffix']
    loss_weight = float(meta['loss_weight'])
    cost_loss = meta['cost_loss']
    rank_loss = meta['rank_loss']

    cost_only_match = re.search(r'costonly_(MSE|MAE|Huber)$', suffix)
    rank_only_match = re.search(r'rankonly_(RankNet|ListNet|LambdaRank)$', suffix)
    combined_match = re.search(r'lw[0-9.]+_(MSE|MAE|Huber)_(RankNet|ListNet|LambdaRank)$', suffix)

    if cost_only_match or np.isclose(loss_weight, 1.0):
        return {'method': f'{(cost_only_match.group(1) if cost_only_match else cost_loss)} cost-only', 'objective_type': 'cost_only'}
    if rank_only_match or np.isclose(loss_weight, 0.0):
        return {'method': f'{(rank_only_match.group(1) if rank_only_match else rank_loss)} rank-only', 'objective_type': 'rank_only'}
    if combined_match:
        return {'method': f"{combined_match.group(1)}+{combined_match.group(2)} lw={loss_weight:g}", 'objective_type': 'cost_rank'}
    return {'method': f'{cost_loss}+{rank_loss} lw={loss_weight:g}', 'objective_type': 'cost_rank'}


def selector_name_from_folder(folder_name: str) -> str:
    # Cost-only folders, e.g. ..._skewed700300_costonly_MAE or ..._stratified_costonly_MAE
    cost_only_match = re.search(r'_(?:skewed700300|stratified)_costonly_(MSE|MAE|Huber)$', folder_name)
    if cost_only_match:
        return f'{cost_only_match.group(1)} cost-only'

    # Rank-only folders, e.g. ..._skewed700300_rankonly_RankNet
    rank_only_match = re.search(r'_(?:skewed700300|stratified)_rankonly_(RankNet|ListNet|LambdaRank)$', folder_name)
    if rank_only_match:
        return f'{rank_only_match.group(1)} rank-only'

    # Combined cost+rank folders, e.g. ..._skewed700300_lw0.5_MSE_LambdaRank
    combined_match = re.search(
        r'_(?:skewed700300|stratified)_lw(?P<weight>[0-9.]+)_(?P<cost>MSE|MAE|Huber)_(?P<rank>RankNet|ListNet|LambdaRank)$',
        folder_name,
    )
    if combined_match:
        return f"{combined_match.group('cost')}+{combined_match.group('rank')} lw={combined_match.group('weight')}"

    # Legacy folders, e.g. ..._70_30_lw0.9_MSE_RankNet
    legacy_match = re.search(r'_lw[0-9.]+_(MSE|MAE|Huber)_(RankNet|ListNet|LambdaRank)$', folder_name)
    if legacy_match:
        return f'{legacy_match.group(1)}+{legacy_match.group(2)}'

    return folder_name




def average_tie_ranks(costs, tolerance=1e-12):
    """Return ascending average ranks from costs using standard tie handling.

    Example: if three solvers tie for the smallest cost, they occupy
    positions 1, 2, and 3, so each receives rank 2. The next distinct
    cost receives rank 4, then rank 5, etc. The rank sum is always
    1 + 2 + ... + n.
    """
    costs = np.asarray(costs, dtype=float)
    order = np.argsort(costs, kind='mergesort')
    ranks = np.empty(len(costs), dtype=float)
    position = 1
    i = 0
    while i < len(order):
        j = i + 1
        while j < len(order) and abs(costs[order[j]] - costs[order[i]]) <= tolerance:
            j += 1
        tied_positions = np.arange(position, position + (j - i), dtype=float)
        ranks[order[i:j]] = tied_positions.mean()
        position += j - i
        i = j
    return ranks

def load_predictions(path: Path) -> pd.DataFrame:
    with path.open() as handle:
        records = json.load(handle)

    selector = selector_name_from_folder(path.parent.name)
    rows = []
    for record in records:
        pred_costs = np.asarray(record['pred_costs'], dtype=float)
        label_costs = np.asarray(record['label_costs'], dtype=float)
        label_ranks = average_tie_ranks(label_costs)
        selected_index = int(np.argmin(pred_costs))
        best_index = int(np.argmin(label_costs))
        rows.append({
            'selector': selector,
            'instance_id': record['instance_id'],
            'selected_solver': SOLVERS[selected_index],
            'selected_solver_index': selected_index,
            'selected_cost': label_costs[selected_index],
            'selected_rank': label_ranks[selected_index],
            'vbs_solver': SOLVERS[best_index],
            'vbs_solver_index': best_index,
            'vbs_cost': label_costs[best_index],
            'vbs_rank': label_ranks[best_index],
            'label_costs': label_costs,
            'label_ranks': label_ranks,
        })
    return pd.DataFrame(rows)


prediction_paths = sorted(RESULTS_ROOT.glob(f'*/{PREDICTION_FILE}'))
print(f'Found {len(prediction_paths)} prediction files')
for path in prediction_paths:
    print(selector_name_from_folder(path.parent.name), '->', path)

selector_df = pd.concat([load_predictions(path) for path in prediction_paths], ignore_index=True)
instances_by_selector = selector_df.groupby('selector')['instance_id'].apply(set)
assert all(instances == instances_by_selector.iloc[0] for instances in instances_by_selector), 'Selectors do not use the same test set.'
print(f'Test instances: {len(instances_by_selector.iloc[0])}')

In [ ]:
selector_summary = (
    selector_df.groupby('selector')
    .agg(
        n_instances=('instance_id', 'count'),
        cost_mean=('selected_cost', 'mean'),
        cost_median=('selected_cost', 'median'),
        cost_std=('selected_cost', 'std'),
        rank_mean=('selected_rank', 'mean'),
        rank_median=('selected_rank', 'median'),
        rank_std=('selected_rank', 'std'),
    )
    .reset_index()
    .sort_values(['cost_mean', 'rank_mean'], kind='mergesort')
)

if BEST_SELECTOR is None:
    BEST_SELECTOR = selector_summary.iloc[0]['selector']

print(f'Selected overall best selector used for both metrics: {BEST_SELECTOR}')
display(selector_summary)


In [ ]:
# Build one true-label table per instance, then compute metric-specific SBS baselines.
label_df = (
    selector_df.sort_values(['selector', 'instance_id'])
    .drop_duplicates('instance_id')
    [['instance_id', 'label_costs', 'label_ranks', 'vbs_solver', 'vbs_cost', 'vbs_rank']]
    .copy()
)

solver_rows = []
for _, row in label_df.iterrows():
    for solver_index, solver in enumerate(SOLVERS):
        solver_rows.append({
            'instance_id': row['instance_id'],
            'solver': solver,
            'cost': row['label_costs'][solver_index],
            'rank': row['label_ranks'][solver_index],
        })

solver_df = pd.DataFrame(solver_rows)
cost_sbs_solver = solver_df.groupby('solver')['cost'].mean().sort_values(kind='mergesort').index[0]
rank_sbs_solver = solver_df.groupby('solver')['rank'].mean().sort_values(kind='mergesort').index[0]
print(f'Cost-SBS solver selected by lowest mean true cost: {cost_sbs_solver}')
print(f'Rank-SBS solver selected by lowest mean true rank: {rank_sbs_solver}')

cost_sbs_df = (
    solver_df[solver_df['solver'] == cost_sbs_solver]
    .rename(columns={'cost': 'cost_sbs_cost', 'rank': 'cost_sbs_rank'})
    [['instance_id', 'cost_sbs_cost', 'cost_sbs_rank']]
)

rank_sbs_df = (
    solver_df[solver_df['solver'] == rank_sbs_solver]
    .rename(columns={'cost': 'rank_sbs_cost', 'rank': 'rank_sbs_rank'})
    [['instance_id', 'rank_sbs_cost', 'rank_sbs_rank']]
)

best_df = (
    selector_df[selector_df['selector'] == BEST_SELECTOR]
    .rename(columns={
        'selected_solver': 'selector_solver',
        'selected_cost': 'selector_cost',
        'selected_rank': 'selector_rank',
    })
    [['instance_id', 'selector_solver', 'selector_cost', 'selector_rank', 'vbs_solver', 'vbs_cost', 'vbs_rank']]
)

analysis_df = best_df.merge(cost_sbs_df, on='instance_id', how='inner').merge(rank_sbs_df, on='instance_id', how='inner')
analysis_df['cost_sbs_solver'] = cost_sbs_solver
analysis_df['rank_sbs_solver'] = rank_sbs_solver
assert len(analysis_df) == len(best_df)
analysis_df.head()


In [ ]:
def add_metric_columns(df, metric, selector_col, sbs_col, vbs_col):
    df[f'{metric}_selector_vbs_gap'] = df[selector_col] - df[vbs_col]
    df[f'{metric}_sbs_vbs_gap'] = df[sbs_col] - df[vbs_col]
    df[f'{metric}_selector_vs_sbs_delta'] = df[selector_col] - df[sbs_col]

    denominator = df[f'{metric}_sbs_vbs_gap'].where(df[f'{metric}_sbs_vbs_gap'].abs() > 0, np.nan)
    stable_denominator = df[f'{metric}_sbs_vbs_gap'].where(df[f'{metric}_sbs_vbs_gap'].abs() > GAP_EPS, np.nan)
    df[f'{metric}_vbs_gap_closed_ratio'] = (df[sbs_col] - df[selector_col]) / denominator
    df[f'{metric}_vbs_gap_closed_ratio_stable'] = (df[sbs_col] - df[selector_col]) / stable_denominator


add_metric_columns(analysis_df, 'cost', 'selector_cost', 'cost_sbs_cost', 'vbs_cost')
add_metric_columns(analysis_df, 'rank', 'selector_rank', 'rank_sbs_rank', 'vbs_rank')

# Backward-compatible aliases for older cells/scripts that expected one cost-SBS analysis.
analysis_df['sbs_cost'] = analysis_df['cost_sbs_cost']
analysis_df['sbs_rank'] = analysis_df['rank_sbs_rank']
analysis_df['selector_vbs_gap'] = analysis_df['cost_selector_vbs_gap']
analysis_df['sbs_vbs_gap'] = analysis_df['cost_sbs_vbs_gap']
analysis_df['selector_vs_sbs_delta'] = analysis_df['cost_selector_vs_sbs_delta']
analysis_df['rank_delta_vs_sbs'] = analysis_df['rank_selector_vs_sbs_delta']
analysis_df['vbs_gap_closed_ratio'] = analysis_df['cost_vbs_gap_closed_ratio']
analysis_df['vbs_gap_closed_ratio_stable'] = analysis_df['cost_vbs_gap_closed_ratio_stable']


def metric_gap_summary(metric, selector_col, sbs_col, vbs_col, sbs_solver):
    selector_vbs_gap = analysis_df[f'{metric}_selector_vbs_gap']
    sbs_vbs_gap = analysis_df[f'{metric}_sbs_vbs_gap']
    ratio = analysis_df[f'{metric}_vbs_gap_closed_ratio']
    ratio_stable = analysis_df[f'{metric}_vbs_gap_closed_ratio_stable']
    delta = analysis_df[f'{metric}_selector_vs_sbs_delta']
    non_tie = ratio.dropna()
    non_tie_stable = ratio_stable.dropna()
    mean_selector = analysis_df[selector_col].mean()
    mean_sbs = analysis_df[sbs_col].mean()
    mean_vbs = analysis_df[vbs_col].mean()
    aggregate_denominator = mean_sbs - mean_vbs
    aggregate_gap_closed_ratio = np.nan if np.isclose(aggregate_denominator, 0) else (mean_sbs - mean_selector) / aggregate_denominator

    return {
        'metric': metric,
        'selector': BEST_SELECTOR,
        'sbs': sbs_solver,
        'n_instances': len(analysis_df),
        'selector_mean': mean_selector,
        'sbs_mean': mean_sbs,
        'vbs_mean': mean_vbs,
        'selector_vbs_gap_mean': selector_vbs_gap.mean(),
        'selector_vbs_gap_median': selector_vbs_gap.median(),
        'selector_vbs_gap_std': selector_vbs_gap.std(),
        'sbs_vbs_gap_mean': sbs_vbs_gap.mean(),
        'sbs_vbs_tie_count': int(ratio.isna().sum()),
        'sbs_vbs_tie_rate': float(ratio.isna().mean()),
        'n_nonzero_sbs_vbs_gap': int(ratio.notna().sum()),
        'n_stable_sbs_vbs_gap': int(ratio_stable.notna().sum()),
        'mean_gap_closed_ratio': non_tie.mean(),
        'median_gap_closed_ratio': non_tie.median(),
        'mean_gap_closed_ratio_stable': non_tie_stable.mean(),
        'median_gap_closed_ratio_stable': non_tie_stable.median(),
        'aggregate_gap_closed_ratio': aggregate_gap_closed_ratio,
        'ratio_eq_1_rate': float(np.isclose(non_tie, 1.0).mean()) if len(non_tie) else np.nan,
        'ratio_between_0_1_rate': float(((non_tie > 0) & (non_tie < 1)).mean()) if len(non_tie) else np.nan,
        'ratio_eq_0_rate': float(np.isclose(non_tie, 0.0).mean()) if len(non_tie) else np.nan,
        'ratio_lt_0_rate': float((non_tie < 0).mean()) if len(non_tie) else np.nan,
        'selector_better_count': int((delta < 0).sum()),
        'tie_count': int(np.isclose(delta, 0).sum()),
        'selector_worse_count': int((delta > 0).sum()),
        'mean_selector_minus_sbs': delta.mean(),
        'median_selector_minus_sbs': delta.median(),
    }


gap_summary = pd.DataFrame([
    metric_gap_summary('cost', 'selector_cost', 'cost_sbs_cost', 'vbs_cost', cost_sbs_solver),
    metric_gap_summary('rank', 'selector_rank', 'rank_sbs_rank', 'vbs_rank', rank_sbs_solver),
])

display(gap_summary)


In [ ]:
pairwise_summary = gap_summary[[
    'metric',
    'selector',
    'sbs',
    'n_instances',
    'selector_better_count',
    'tie_count',
    'selector_worse_count',
    'mean_selector_minus_sbs',
    'median_selector_minus_sbs',
]].copy()

pairwise_summary = pairwise_summary.rename(columns={
    'mean_selector_minus_sbs': 'mean_selector_minus_sbs_value',
    'median_selector_minus_sbs': 'median_selector_minus_sbs_value',
})

display(pairwise_summary)


In [ ]:
# Wilcoxon signed-rank tests. Lower cost/rank is better.
# alternative='less' tests whether the selected overall selector has lower paired values than the metric-specific SBS.
def safe_wilcoxon(x, y, alternative):
    if np.allclose(np.asarray(x) - np.asarray(y), 0):
        return pd.Series({'statistic': 0.0, 'p_value': 1.0})
    result = wilcoxon(x, y, alternative=alternative, zero_method='wilcox')
    return pd.Series({'statistic': result.statistic, 'p_value': result.pvalue})

wilcoxon_rows = []
for metric, selector_col, sbs_col, sbs_solver in [
    ('cost', 'selector_cost', 'cost_sbs_cost', cost_sbs_solver),
    ('rank', 'selector_rank', 'rank_sbs_rank', rank_sbs_solver),
]:
    less = safe_wilcoxon(analysis_df[selector_col], analysis_df[sbs_col], alternative='less')
    two_sided = safe_wilcoxon(analysis_df[selector_col], analysis_df[sbs_col], alternative='two-sided')
    wilcoxon_rows.extend([
        {
            'metric': metric,
            'selector': BEST_SELECTOR,
            'sbs': sbs_solver,
            'alternative': f'{BEST_SELECTOR} < SBS',
            'statistic': less['statistic'],
            'p_value': less['p_value'],
        },
        {
            'metric': metric,
            'selector': BEST_SELECTOR,
            'sbs': sbs_solver,
            'alternative': 'two-sided',
            'statistic': two_sided['statistic'],
            'p_value': two_sided['p_value'],
        },
    ])

wilcoxon_summary = pd.DataFrame(wilcoxon_rows)
display(wilcoxon_summary)


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))

for row, metric in enumerate(['cost', 'rank']):
    delta_col = f'{metric}_selector_vs_sbs_delta'
    gap_col = f'{metric}_selector_vbs_gap'
    ratio_col = f'{metric}_vbs_gap_closed_ratio'

    axes[row, 0].hist(analysis_df[delta_col], bins=40, color='#4C78A8', edgecolor='white')
    axes[row, 0].axvline(0, color='black', linewidth=1)
    axes[row, 0].set_title(f'{BEST_SELECTOR} {metric} - {metric}-SBS {metric}')
    axes[row, 0].set_xlabel(f'Paired {metric} difference')
    axes[row, 0].set_ylabel('Instances')

    axes[row, 1].hist(analysis_df[gap_col], bins=40, color='#59A14F', edgecolor='white')
    axes[row, 1].axvline(0, color='black', linewidth=1)
    axes[row, 1].set_title(f'{BEST_SELECTOR} {metric} VBS gap')
    axes[row, 1].set_xlabel(f'selector {metric} - VBS {metric}')

    axes[row, 2].hist(analysis_df[ratio_col].dropna(), bins=40, color='#F28E2B', edgecolor='white')
    axes[row, 2].axvline(0, color='black', linewidth=1)
    axes[row, 2].axvline(1, color='black', linewidth=1, linestyle='--')
    axes[row, 2].set_title(f'{metric} VBS gap closed ratio')
    axes[row, 2].set_xlabel('closed ratio')

plt.tight_layout()
plt.show()


In [ ]:
selection_counts = pd.DataFrame({
    BEST_SELECTOR: analysis_df['selector_solver'].value_counts().reindex(SOLVERS, fill_value=0),
    f'SBS cost ({cost_sbs_solver})': pd.Series({solver: (len(analysis_df) if solver == cost_sbs_solver else 0) for solver in SOLVERS}),
    f'SBS rank ({rank_sbs_solver})': pd.Series({solver: (len(analysis_df) if solver == rank_sbs_solver else 0) for solver in SOLVERS}),
    'VBS': analysis_df['vbs_solver'].value_counts().reindex(SOLVERS, fill_value=0),
})
display(selection_counts)


In [ ]:
output_dir = RESULTS_ROOT / 'best_selector_analysis'
output_dir.mkdir(exist_ok=True)

analysis_df.to_csv(output_dir / 'per_instance_best_selector_sbs_vbs.csv', index=False)
gap_summary.to_csv(output_dir / 'vbs_gap_summary.csv', index=False)
pairwise_summary.to_csv(output_dir / 'selector_vs_sbs_pairwise_summary.csv', index=False)
wilcoxon_summary.to_csv(output_dir / 'wilcoxon_selector_vs_sbs.csv', index=False)
selection_counts.to_csv(output_dir / 'selection_counts_best_sbs_vbs.csv')

print(f'Wrote analysis tables to: {output_dir.resolve()}')